In [1]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 46.7 MB/s eta 0:00:00


In [2]:
from ultralytics import YOLO

# load your trained model
model = YOLO("/content/drive/MyDrive/yolo_assignment/runs/detect/train/weights/best.pt")

# run prediction WITH labels saving
model.predict(
    source="/content/drive/MyDrive/yolo_assignment/coco_yolo/images/val",
    imgsz=640,            # same as training (important)
    conf=0.25,            # confidence threshold
    iou=0.7,              # NMS IoU threshold
    save=True,            # save images (optional)
    save_txt=True,        # ⭐ MUST (creates label files)
    save_conf=True,       # ⭐ MUST (adds confidence)
    project="/content/drive/MyDrive/yolo_assignment",
    name="predict_final", # folder name (so it doesn't overwrite)
    exist_ok=True,
    device=0              # GPU
)

Streaming output truncated to the last 5000 lines.
image 4/5000 /content/drive/MyDrive/yolo_assignment/coco_yolo/images/val/000000000724.jpg: 640x480 1 truck, 42.6ms
image 5/5000 /content/drive/MyDrive/yolo_assignment/coco_yolo/images/val/000000000776.jpg: 640x448 (no detections), 43.4ms
image 6/5000 /content/drive/MyDrive/yolo_assignment/coco_yolo/images/val/000000000785.jpg: 448x640 1 person, 10.9ms
image 7/5000 /content/drive/MyDrive/yolo_assignment/coco_yolo/images/val/000000000802.jpg: 640x448 (no detections), 9.4ms
image 8/5000 /content/drive/MyDrive/yolo_assignment/coco_yolo/images/val/000000000872.jpg: 640x640 3 persons, 10.7ms
image 9/5000 /content/drive/MyDrive/yolo_assignment/coco_yolo/images/val/000000000885.jpg: 448x640 2 persons, 1 bicycle, 12.0ms
image 10/5000 /content/drive/MyDrive/yolo_assignment/coco_yolo/images/val/000000001000.jpg: 480x640 14 persons, 42.2ms
image 11/5000 /content/drive/MyDrive/yolo_assignment/coco_yolo/images/val/000000001268.jpg: 448x640 5 persons

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'person', 1: 'bicycle', 2: 'car', 3: 'motorcycle', 4: 'airplane', 5: 'bus', 6: 'train', 7: 'truck', 8: 'boat', 9: 'traffic light'}
 obb: None
 orig_img: array([[[ 73, 136, 170],
         [ 77, 142, 173],
         [ 79, 144, 175],
         ...,
         [ 42,  76,  69],
         [ 39,  76,  68],
         [ 37,  71,  70]],
 
        [[ 77, 141, 172],
         [ 80, 145, 176],
         [ 81, 146, 177],
         ...,
         [ 40,  77,  69],
         [ 43,  80,  72],
         [ 40,  75,  71]],
 
        [[ 79, 144, 175],
         [ 81, 146, 177],
         [ 80, 147, 178],
         ...,
         [ 39,  78,  70],
         [ 40,  77,  69],
         [ 40,  75,  71]],
 
        ...,
 
        [[157, 189, 188],
         [149, 183, 183],
         [153, 187, 193],
         ...,
         [153, 157, 186],
         [153, 157, 186],
         [154, 156

In [3]:
pred_dir = "/content/drive/MyDrive/yolo_assignment/predict_final/labels"

In [5]:
!pip install mean-average-precision

In [9]:
gt_files = set(os.listdir(gt_dir))
pred_files = set(os.listdir(pred_dir))

common = gt_files & pred_files

print("GT files:", len(gt_files))
print("Pred files:", len(pred_files))
print("Matching files:", len(common))

GT files: 4952
Pred files: 3693
Matching files: 3692


In [11]:
count = 0
with open(gt_file) as f:
    for line in f:
        cls = int(line.split()[0])
        if cls == 0:
            count += 1

print("Class 0 objects:", count)

Class 0 objects: 1


In [14]:
import os
import numpy as np

pred_dir = "/content/drive/MyDrive/yolo_assignment/predict_final/labels"
gt_dir   = "/content/drive/MyDrive/yolo_assignment/coco_yolo/labels/val"

selected_classes = list(range(10))   # your trained classes
IOU_THRESH = 0.5

def yolo_to_xyxy(x, y, w, h):
    return [x - w/2, y - h/2, x + w/2, y + h/2]

def iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2]-box1[0]) * (box1[3]-box1[1])
    area2 = (box2[2]-box2[0]) * (box2[3]-box2[1])

    union = area1 + area2 - inter
    return inter / union if union > 0 else 0

TP = 0
FP = 0

for file in os.listdir(gt_dir):

    gt_file = os.path.join(gt_dir, file)
    pred_file = os.path.join(pred_dir, file)

    gt_boxes = []
    pred_boxes = []

    # -------- GT --------
    with open(gt_file) as f:
        for line in f:
            cls, x, y, w, h = map(float, line.split())
            if int(cls) in selected_classes:
                gt_boxes.append([int(cls), *yolo_to_xyxy(x,y,w,h)])

    if len(gt_boxes) == 0:
        continue

    # -------- Predictions --------
    if os.path.exists(pred_file):
        with open(pred_file) as f:
            for line in f:
                cls, x, y, w, h, conf = map(float, line.split())
                if int(cls) in selected_classes:
                    pred_boxes.append([int(cls), *yolo_to_xyxy(x,y,w,h)])

    matched = set()

    for pred in pred_boxes:
        p_cls, *p_box = pred
        found_match = False

        for i, gt in enumerate(gt_boxes):
            g_cls, *g_box = gt

            if i in matched:
                continue

            if p_cls == g_cls and iou(p_box, g_box) >= IOU_THRESH:
                TP += 1
                matched.add(i)
                found_match = True
                break

        if not found_match:
            FP += 1

# -------- FINAL PRECISION --------
precision = TP / (TP + FP) if (TP + FP) > 0 else 0

print("TP:", TP)
print("FP:", FP)
print("Precision:", precision)

TP: 9017
FP: 6227
Precision: 0.5915114143269483


In [15]:
import os
import numpy as np

pred_dir = "/content/drive/MyDrive/yolo_assignment/predict_final/labels"
gt_dir   = "/content/drive/MyDrive/yolo_assignment/coco_yolo/labels/val"

selected_classes = list(range(10))
IOU_THRESH = 0.5

def yolo_to_xyxy(x, y, w, h):
    return [x - w/2, y - h/2, x + w/2, y + h/2]

def iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2]-box1[0]) * (box1[3]-box1[1])
    area2 = (box2[2]-box2[0]) * (box2[3]-box2[1])

    union = area1 + area2 - inter
    return inter / union if union > 0 else 0

TP = 0
FN = 0

for file in os.listdir(gt_dir):

    gt_file = os.path.join(gt_dir, file)
    pred_file = os.path.join(pred_dir, file)

    gt_boxes = []
    pred_boxes = []

    # -------- GT --------
    with open(gt_file) as f:
        for line in f:
            cls, x, y, w, h = map(float, line.split())
            if int(cls) in selected_classes:
                gt_boxes.append([int(cls), *yolo_to_xyxy(x,y,w,h)])

    if len(gt_boxes) == 0:
        continue

    # -------- Predictions --------
    if os.path.exists(pred_file):
        with open(pred_file) as f:
            for line in f:
                cls, x, y, w, h, conf = map(float, line.split())
                if int(cls) in selected_classes:
                    pred_boxes.append([int(cls), *yolo_to_xyxy(x,y,w,h)])

    matched = set()

    for i, gt in enumerate(gt_boxes):
        g_cls, *g_box = gt
        found = False

        for pred in pred_boxes:
            p_cls, *p_box = pred

            if p_cls == g_cls and iou(p_box, g_box) >= IOU_THRESH:
                found = True
                break

        if found:
            TP += 1
        else:
            FN += 1

# -------- FINAL RECALL --------
recall = TP / (TP + FN) if (TP + FN) > 0 else 0

print("TP:", TP)
print("FN:", FN)
print("Recall:", recall)

TP: 9096
FN: 6368
Recall: 0.5882048629073978


In [16]:
import os
import numpy as np

pred_dir = "/content/drive/MyDrive/yolo_assignment/predict_final/labels"
gt_dir   = "/content/drive/MyDrive/yolo_assignment/coco_yolo/labels/val"

selected_classes = list(range(10))
IOU_THRESH = 0.5

def yolo_to_xyxy(x, y, w, h):
    return [x - w/2, y - h/2, x + w/2, y + h/2]

def iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2]-box1[0]) * (box1[3]-box1[1])
    area2 = (box2[2]-box2[0]) * (box2[3]-box2[1])

    union = area1 + area2 - inter
    return inter / union if union > 0 else 0

# store per class results
class_tp = {c: [] for c in selected_classes}
class_fp = {c: [] for c in selected_classes}
class_scores = {c: [] for c in selected_classes}
class_gt_count = {c: 0 for c in selected_classes}

# ---------- LOOP ----------
for file in os.listdir(gt_dir):

    gt_file = os.path.join(gt_dir, file)
    pred_file = os.path.join(pred_dir, file)

    gt_boxes = []

    # GT
    with open(gt_file) as f:
        for line in f:
            cls, x, y, w, h = map(float, line.split())
            if int(cls) in selected_classes:
                gt_boxes.append([int(cls), *yolo_to_xyxy(x,y,w,h)])
                class_gt_count[int(cls)] += 1

    if len(gt_boxes) == 0:
        continue

    used = set()

    # Predictions
    if os.path.exists(pred_file):
        preds = []
        with open(pred_file) as f:
            for line in f:
                cls, x, y, w, h, conf = map(float, line.split())
                if int(cls) in selected_classes:
                    preds.append([int(cls), conf, *yolo_to_xyxy(x,y,w,h)])

        # sort by confidence
        preds.sort(key=lambda x: x[1], reverse=True)

        for pred in preds:
            p_cls, conf, *p_box = pred
            best_iou = 0
            best_gt = -1

            for i, gt in enumerate(gt_boxes):
                g_cls, *g_box = gt

                if i in used or g_cls != p_cls:
                    continue

                iou_val = iou(p_box, g_box)
                if iou_val > best_iou:
                    best_iou = iou_val
                    best_gt = i

            if best_iou >= IOU_THRESH:
                class_tp[p_cls].append(1)
                class_fp[p_cls].append(0)
                used.add(best_gt)
            else:
                class_tp[p_cls].append(0)
                class_fp[p_cls].append(1)

            class_scores[p_cls].append(conf)

# ---------- COMPUTE AP ----------
aps = []

for c in selected_classes:

    tp = np.array(class_tp[c])
    fp = np.array(class_fp[c])

    if len(tp) == 0:
        continue

    # cumulative
    tp_cum = np.cumsum(tp)
    fp_cum = np.cumsum(fp)

    recalls = tp_cum / (class_gt_count[c] + 1e-6)
    precisions = tp_cum / (tp_cum + fp_cum + 1e-6)

    # AP = area under curve
    ap = np.trapz(precisions, recalls)
    aps.append(ap)

# ---------- FINAL mAP ----------
mAP = np.mean(aps) if len(aps) > 0 else 0

print("mAP@0.5:", mAP)

mAP@0.5: 0.2700482990931522


/tmp/ipykernel_3507/3085068553.py:110: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  ap = np.trapz(precisions, recalls)


In [17]:
import os
import numpy as np

pred_dir = "/content/drive/MyDrive/yolo_assignment/predict_final/labels"
gt_dir   = "/content/drive/MyDrive/yolo_assignment/coco_yolo/labels/val"

selected_classes = list(range(10))

# IoU thresholds (COCO style)
iou_thresholds = np.arange(0.5, 1.0, 0.05)

def yolo_to_xyxy(x, y, w, h):
    return [x - w/2, y - h/2, x + w/2, y + h/2]

def iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2]-box1[0]) * (box1[3]-box1[1])
    area2 = (box2[2]-box2[0]) * (box2[3]-box2[1])

    union = area1 + area2 - inter
    return inter / union if union > 0 else 0

all_maps = []

# ---------- LOOP OVER IOU THRESHOLDS ----------
for IOU_THRESH in iou_thresholds:

    class_tp = {c: [] for c in selected_classes}
    class_fp = {c: [] for c in selected_classes}
    class_scores = {c: [] for c in selected_classes}
    class_gt_count = {c: 0 for c in selected_classes}

    for file in os.listdir(gt_dir):

        gt_file = os.path.join(gt_dir, file)
        pred_file = os.path.join(pred_dir, file)

        gt_boxes = []

        # -------- GT --------
        with open(gt_file) as f:
            for line in f:
                cls, x, y, w, h = map(float, line.split())
                if int(cls) in selected_classes:
                    gt_boxes.append([int(cls), *yolo_to_xyxy(x,y,w,h)])
                    class_gt_count[int(cls)] += 1

        if len(gt_boxes) == 0:
            continue

        used = set()

        # -------- Predictions --------
        if os.path.exists(pred_file):
            preds = []
            with open(pred_file) as f:
                for line in f:
                    cls, x, y, w, h, conf = map(float, line.split())
                    if int(cls) in selected_classes:
                        preds.append([int(cls), conf, *yolo_to_xyxy(x,y,w,h)])

            preds.sort(key=lambda x: x[1], reverse=True)

            for pred in preds:
                p_cls, conf, *p_box = pred
                best_iou = 0
                best_gt = -1

                for i, gt in enumerate(gt_boxes):
                    g_cls, *g_box = gt

                    if i in used or g_cls != p_cls:
                        continue

                    iou_val = iou(p_box, g_box)
                    if iou_val > best_iou:
                        best_iou = iou_val
                        best_gt = i

                if best_iou >= IOU_THRESH:
                    class_tp[p_cls].append(1)
                    class_fp[p_cls].append(0)
                    used.add(best_gt)
                else:
                    class_tp[p_cls].append(0)
                    class_fp[p_cls].append(1)

                class_scores[p_cls].append(conf)

    # ---------- AP per class ----------
    aps = []

    for c in selected_classes:

        tp = np.array(class_tp[c])
        fp = np.array(class_fp[c])

        if len(tp) == 0:
            continue

        tp_cum = np.cumsum(tp)
        fp_cum = np.cumsum(fp)

        recalls = tp_cum / (class_gt_count[c] + 1e-6)
        precisions = tp_cum / (tp_cum + fp_cum + 1e-6)

        ap = np.trapz(precisions, recalls)
        aps.append(ap)

    if len(aps) > 0:
        all_maps.append(np.mean(aps))

# ---------- FINAL RESULT ----------
map_5095 = np.mean(all_maps) if len(all_maps) > 0 else 0

print("mAP@0.5:0.95:", map_5095)

/tmp/ipykernel_3507/656810549.py:112: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  ap = np.trapz(precisions, recalls)


mAP@0.5:0.95: 0.1774687188845138
